<a href="https://colab.research.google.com/github/office268/ex2/blob/main/voice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# התקנת ספריות נדרשות
!pip install -q diffusers transformers accelerate scipy
!pip install -q --upgrade sympy

import torch
from diffusers import AudioLDM2Pipeline
from IPython.display import Audio
import scipy

# 1. טעינת המודל (דורש GPU ב-Colab: Runtime -> Change runtime type -> T4 GPU)
pipe = AudioLDM2Pipeline.from_pretrained("cvssp/audioldm2-music", torch_dtype=torch.float16)
pipe.to("cuda")

# 2. הגדרת הפרומפט (התיאור המוזיקלי)
prompt = "High-quality cinematic fusion: Deep Techno beat 124 BPM, sharp electronic percussion, layered with a clear solo Oud melody. Rich bass, crisp high-end frequencies, professional studio mastering, 44.1kHz, 8k resolution audio, wide stereo image, immersive desert atmosphere, no muffled sounds"
negative_prompt = "low quality, average quality, noise, distortion"

# 3. יצירת האודיו
print("מייצר מוזיקה... זה ייקח רגע...")
audio = pipe(
    prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=200,
    audio_length_in_s=10,
    guidance_scale=3.5
).audios[0]

# 4. שמירה לקובץ WAV
scipy.io.wavfile.write("generated_music.wav", rate=16000, data=audio)
print("נשמר כ-generated_music.wav")

# 5. השמעה
Audio(audio, rate=16000)

In [ ]:
import librosa
import numpy as np
import matplotlib.pyplot as plt

def evaluate_audio_quality(audio_data, sampling_rate=16000):
    print("--- ניתוח איכות אובייקטיבי ---")

    # 1. ניתוח קצב (Tempo Detection)
    # בודק אם המודל הצליח לשמור על ביט עקבי
    tempo, _ = librosa.beat.beat_track(y=audio_data, sr=sampling_rate)
    print(f"1. קצב מזוהה: {tempo.item():.2f} BPM")

    # 2. ניתוח בהירות (Spectral Centroid)
    # בודק אם הסאונד "עמום" או "צלול"
    cent = librosa.feature.spectral_centroid(y=audio_data, sr=sampling_rate)
    mean_cent = np.mean(cent)
    print(f"2. מרכז ספקטרלי (בהירות): {mean_cent:.2f} Hz")
    # (ערך גבוה יותר = סאונד צלול יותר ופחות "בוצי")

    # 3. ניתוח עוצמה (RMS Energy)
    # בודק אם יש תנודות ווליום לא רצויות
    rms = librosa.feature.rms(y=audio_data)
    print(f"3. יציבות עוצמה (RMS): {np.std(rms):.4f} (סטיית תקן)")

    # ויזואליזציה של הספקטרוגרמה
    plt.figure(figsize=(10, 4))
    D = librosa.amplitude_to_db(np.abs(librosa.stft(audio_data)), ref=np.max)
    librosa.display.specshow(D, sr=sampling_rate, x_axis='time', y_axis='hz')
    plt.colorbar(format='%+2.0f dB')
    plt.title('Spectrogram: בדיקת פיזור תדרים')
    plt.show()

# הרצה על האודיו שיצרת בתא הקודם
evaluate_audio_quality(audio)